# Encrypted fairness metrics with `fairlearn-fhe`

This notebook mirrors the existing [fairness-dashboard-loan-allocation](fairness-dashboard-loan-allocation.ipynb) walkthrough but computes the per-group rates under **CKKS homomorphic encryption** via [`fairlearn-fhe`](https://github.com/BAder82t/fairlearn-fhe). The drop-in replacement keeps the Fairlearn API surface; only the import line changes:

```python
# plaintext (status quo)
from fairlearn.metrics import demographic_parity_difference

# encrypted (one import change)
from fairlearn_fhe.metrics import demographic_parity_difference
```

The encrypted verdicts agree with the plaintext baseline within CKKS noise tolerance (`< 1e-3` absolute error in default settings), and the run produces a tamper-evident `MetricEnvelope` an auditor can hand to a regulator.

## Install required packages

In [ ]:
# %pip install --upgrade fairlearn-fhe fairlearn raiwidgets

## Load the UCI Adult dataset and train a baseline classifier

Identical setup to the existing fairness-dashboard notebook so the encrypted verdicts can be cross-checked directly against the plaintext ones.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = fetch_openml(data_id=1590, as_frame=True)
X_raw = data.data
y_true = (data.target == ">50K").astype(int)
sensitive = X_raw["sex"].astype(str)

X = X_raw.drop(columns=["sex"])
X_train, X_test, y_train, y_test, sf_train, sf_test = train_test_split(
    X, y_true, sensitive, test_size=0.3, random_state=12345, stratify=y_true
)

numeric = X_train.select_dtypes(include="number").columns
categorical = X_train.select_dtypes(exclude="number").columns
preprocess = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ]
)
model = Pipeline(
    [("prep", preprocess), ("clf", LogisticRegression(max_iter=1000))]
).fit(X_train, y_train)
y_pred = model.predict(X_test)

len(y_pred), float(y_pred.mean())

## Plaintext baseline (Fairlearn)

Compute the canonical fairness metrics in plaintext so we have a reference for the encrypted run.

In [ ]:
from fairlearn.metrics import (
    demographic_parity_difference,
    equalized_odds_difference,
    equal_opportunity_difference,
)

plain = {
    "demographic_parity_difference": demographic_parity_difference(
        y_test, y_pred, sensitive_features=sf_test
    ),
    "equalized_odds_difference": equalized_odds_difference(
        y_test, y_pred, sensitive_features=sf_test
    ),
    "equal_opportunity_difference": equal_opportunity_difference(
        y_test, y_pred, sensitive_features=sf_test
    ),
}
plain

## Encrypted run via `fairlearn-fhe`

Build a CKKS context, encrypt `y_pred`, and call the same-named helpers from `fairlearn_fhe.metrics`. The auditor still holds plaintext `y_test` and `sf_test`; only the model's predictions cross the trust boundary as ciphertext.

This is **Mode A** in the `fairlearn-fhe` [trust-models guide](https://github.com/BAder82t/fairlearn-fhe/blob/main/docs/trust-models.md): encrypted predictions, plaintext sensitive features, depth-1 CKKS circuit per metric.

In [ ]:
from fairlearn_fhe import build_context, encrypt
from fairlearn_fhe.metrics import (
    demographic_parity_difference as enc_dp,
    equalized_odds_difference as enc_eo,
    equal_opportunity_difference as enc_eopp,
)

ctx = build_context(backend="tenseal")
y_pred_enc = encrypt(ctx, y_pred.astype(float))

encrypted = {
    "demographic_parity_difference": enc_dp(
        y_test, y_pred_enc, sensitive_features=sf_test
    ),
    "equalized_odds_difference": enc_eo(
        y_test, y_pred_enc, sensitive_features=sf_test
    ),
    "equal_opportunity_difference": enc_eopp(
        y_test, y_pred_enc, sensitive_features=sf_test
    ),
}
encrypted

## Numerical agreement check

Encrypted verdicts should match the plaintext baseline within CKKS noise tolerance.

In [ ]:
TOL = 1e-3
for name in plain:
    delta = abs(plain[name] - encrypted[name])
    flag = "OK" if delta < TOL else "!!"
    print(f"{flag} {name}: plain={plain[name]:.6f} fhe={encrypted[name]:.6f} |Δ|={delta:.2e}")

## Audit envelope

`audit_metric` packages the verdict + parameter-set hash + observed depth + op counts + input hashes + a UTC timestamp into a `MetricEnvelope`. The envelope is JSON-serialisable, validates against the published [JSON Schema](https://github.com/BAder82t/fairlearn-fhe/blob/main/docs/api/envelope.md), and can be Ed25519-signed for regulator handoff.

In [ ]:
from fairlearn_fhe import audit_metric

env = audit_metric(
    "demographic_parity_difference",
    y_true=y_test,
    y_pred=y_pred_enc,
    sensitive_features=sf_test,
    ctx=ctx,
    min_group_size=30,
)
env_dict = env.to_dict()
{k: env_dict[k] for k in [
    "metric_name", "value", "observed_depth", "op_counts", "n_samples", "n_groups", "trust_model"
]}

Verify the envelope from the command line (no FHE backend required by the verifier):

```bash
fairlearn-fhe verify envelope.json --max-age 86400 --min-security-bits 128
```

## Hand off the same predictions to the Fairness dashboard

Decrypted per-group rates can flow into the existing `raiwidgets.FairnessDashboard` for interactive exploration; the encrypted verdict is the regulator-facing artefact, the dashboard is the analyst-facing one.

In [ ]:
# from raiwidgets import FairnessDashboard
# FairnessDashboard(
#     sensitive_features=sf_test,
#     y_true=y_test,
#     y_pred={"baseline": y_pred},
# )

## Trust models recap

| Mode | Encrypted | Plaintext | Depth |
| --- | --- | --- | --- |
| A (default) | `y_pred` | `y_true`, `sensitive_features`, group counts | 1 |
| B (full encryption) | `y_pred`, per-row group masks | `y_true`, group counts | 2 |

Mode B is described in `fairlearn_fhe.encrypt_sensitive_features`; see [trust-models.md](https://github.com/BAder82t/fairlearn-fhe/blob/main/docs/trust-models.md) for the cost / privacy tradeoff. The included [threat-model](https://bader82t.github.io/fairlearn-fhe/threat-model/) page formalises what the auditor learns vs does not learn.